# 172. MCP Client Features：Sampling、Elicitation、Roots 与能力协商怎样实现？

> **面试问题：MCP server 如何通过 client 请求模型采样、用户补充信息和 roots？怎样做版本协商、审批与有界 agent loop？**

## 先给结论

Sampling、Elicitation 和 Roots 是 MCP client 提供给 server 的可选能力，必须在 initialize 中协商后才能使用。Sampling 的模型凭证与最终控制留在 client/host；Elicitation 要区分 form 与 URL 模式；Roots 只是 server 可询问的边界提示，真正文件授权仍由 host 强制执行。每次请求都需审批、预算、schema、审计和取消。

## 推荐回答主线

1. 先区分 Host、Client、Server，交换 protocolVersion 与 capability；未协商能力直接拒绝。
2. 实现 JSON-RPC request id、sampling/createMessage、roots/list 和 elicitation/create 的最小合同。
3. Sampling tool loop 设置轮次/token/tool/成本预算，client 选择模型并持有凭证，危险动作回到用户审批。
4. 绑定协议修订；处理 2025-11-25 的 URL elicitation、sampling tools，以及后续 draft/弃用变化。

## 教学边界

这是协议状态机与 host policy 的离线模拟，不启动网络 server，也不替代官方 SDK。示例以已发布的 2025-11-25 修订为基线；生产必须按实际协商版本读取对应 schema，并追踪最新规范/SEP。

## 一手资料

- [MCP 2025-11-25 Lifecycle](https://modelcontextprotocol.io/specification/2025-11-25/basic/lifecycle)
- [MCP Sampling](https://modelcontextprotocol.io/specification/2025-11-25/client/sampling)
- [MCP Elicitation](https://modelcontextprotocol.io/specification/2025-11-25/client/elicitation)
- [MCP Schema / Roots](https://modelcontextprotocol.io/specification/2025-11-25/schema)


In [ ]:
import hashlib
import json
import posixpath
import re
from dataclasses import dataclass, field
from urllib.parse import unquote, urlsplit

# 固定已发布修订；线上不能把 draft schema 当作已协商稳定接口。
PROTOCOL_VERSION = "2025-11-25"
client_capabilities = {
    "roots": {"listChanged": True},
    "sampling": {"tools": {}},
    "elicitation": {"form": {}, "url": {}},
}

assert re.fullmatch(r"\d{4}-\d{2}-\d{2}", PROTOCOL_VERSION)
assert "sampling" in client_capabilities
assert "url" in client_capabilities["elicitation"]


## 1. Initialize：版本与能力是协商结果，不是 server 单方面宣称

client 首先发送 initialize；server 返回它支持的版本/能力，client 若不支持返回版本应断开。只有双方交集可用于会话。能力树要按子能力检查，例如 sampling 存在不代表支持 sampling tools。


In [ ]:
def negotiate(client_version, client_caps, server_versions, server_caps):
    if client_version not in server_versions:
        return {"ok": False, "reason": "unsupported_protocol"}
    common = {name: value for name, value in client_caps.items() if name in server_caps}
    return {"ok": True, "version": client_version, "capabilities": common}

# 同版本成功且只保留交集；无共同版本失败；server 未声明的能力不能使用。
session = negotiate(PROTOCOL_VERSION, client_capabilities, {PROTOCOL_VERSION}, {"sampling": {}, "elicitation": {}, "tools": {}})
assert session["ok"] and session["version"] == PROTOCOL_VERSION
assert set(session["capabilities"]) == {"sampling", "elicitation"}
assert not negotiate(PROTOCOL_VERSION, client_capabilities, {"2025-06-18"}, {})["ok"]


## 2. JSON-RPC envelope：request、notification 与 response 不能混淆

MCP 消息遵循 JSON-RPC 2.0。request 有 id 并期待 response；notification 无 id；response 必须二选一包含 result 或 error。id 在未决请求中唯一，取消也绑定原 request id。


In [ ]:
def validate_jsonrpc(message):
    if message.get("jsonrpc") != "2.0":
        return False, "bad_version"
    if "method" in message:
        return (isinstance(message["method"], str), "request_or_notification")
    if "id" not in message or (("result" in message) == ("error" in message)):
        return False, "bad_response"
    return True, "response"

# 合法请求和结果通过；同时带 result/error 或缺 id 的 response 被拒绝。
request = {"jsonrpc": "2.0", "id": 7, "method": "roots/list", "params": {}}
assert validate_jsonrpc(request)[0]
assert validate_jsonrpc({"jsonrpc": "2.0", "id": 7, "result": {}})[0]
assert not validate_jsonrpc({"jsonrpc": "2.0", "id": 7, "result": {}, "error": {}})[0]


## 3. Roots：列出允许上下文，但 host 仍要强制路径边界

roots/list 返回 `file://` URI。规范中的 roots 帮 server 理解工作区，不等于操作系统授权；host 仍要规范化 percent encoding、拒绝 traversal/symlink 逃逸，并在真正读取时校验 capability。这里演示纯 URI 边界。


In [ ]:
def normalized_file_path(uri):
    parsed = urlsplit(uri)
    if parsed.scheme != "file" or parsed.netloc not in {"", "localhost"}:
        raise ValueError("只接受本地 file URI")
    decoded = unquote(parsed.path)
    return posixpath.normpath(decoded)

def uri_within_roots(uri, roots):
    target = normalized_file_path(uri)
    for root in roots:
        base = normalized_file_path(root).rstrip("/")
        if target == base or target.startswith(base + "/"):
            return True
    return False

# 子路径允许；`..` 与 percent 编码 traversal 规范化后越界；前缀相似目录不算子目录。
roots = ["file:///workspace/project"]
assert uri_within_roots("file:///workspace/project/src/a.py", roots)
assert not uri_within_roots("file:///workspace/project/../secret.txt", roots)
assert not uri_within_roots("file:///workspace/project-evil/x", roots)


## 4. Sampling：server 提需求，client 选模型、持凭证并审批

sampling/createMessage 可带 messages、maxTokens、modelPreferences，以及 2025-11-25 的 tools/toolChoice。server 不需要模型 API key；client 可修改/拒绝 prompt、选择不同模型并决定返回哪些内容。未声明 `sampling.tools` 时不得发送 tools。


In [ ]:
def authorize_sampling(params, capabilities, approved, remaining_tokens):
    if "sampling" not in capabilities:
        return False, "capability_missing"
    if params.get("tools") and "tools" not in capabilities["sampling"]:
        return False, "sampling_tools_not_supported"
    if not approved:
        return False, "user_declined"
    max_tokens = int(params.get("maxTokens", 0))
    if max_tokens <= 0 or max_tokens > remaining_tokens:
        return False, "token_budget"
    return True, "approved"

# 已协商且预算内才通过；用户拒绝和超预算都有明确原因。
sample_params = {"messages": [{"role": "user", "content": {"type": "text", "text": "总结结果"}}], "maxTokens": 80, "tools": [{"name": "lookup"}]}
assert authorize_sampling(sample_params, client_capabilities, True, 100)[0]
assert authorize_sampling(sample_params, client_capabilities, False, 100)[1] == "user_declined"
assert authorize_sampling(sample_params, client_capabilities, True, 20)[1] == "token_budget"


## 5. 有界 Sampling tool loop：递归 agent 行为必须有多维预算

sampling 支持 tools 后，server 可请求 client 模型在采样中调用工具。client 仍应限制轮次、token、tool calls、wall time 和重复签名，并逐次执行权限检查。下面用 mock model 演示 call→result→final。


In [ ]:
def run_sampling_loop(model_steps, tools, max_rounds=3, max_tool_calls=2):
    trace, tool_calls = [], 0
    for round_id, step in enumerate(model_steps):
        if round_id >= max_rounds:
            return {"status": "budget_exhausted", "trace": trace}
        if step["type"] == "final":
            trace.append(step)
            return {"status": "completed", "trace": trace, "text": step["text"]}
        if step["type"] != "tool_call" or step["name"] not in tools:
            return {"status": "invalid_step", "trace": trace}
        tool_calls += 1
        if tool_calls > max_tool_calls:
            return {"status": "budget_exhausted", "trace": trace}
        trace.extend([step, {"type": "tool_result", "content": tools[step["name"]](step["args"])}])
    return {"status": "no_final", "trace": trace}

# 正常轨迹完成；重复工具超过预算终止；未知工具拒绝而非让模型猜。
steps = [{"type": "tool_call", "name": "lookup", "args": {"id": 2}}, {"type": "final", "text": "记录 2 已找到"}]
result = run_sampling_loop(steps, {"lookup": lambda args: {"id": args["id"]}})
assert result["status"] == "completed"
assert run_sampling_loop([steps[0]] * 3, {"lookup": lambda args: args})["status"] == "budget_exhausted"
assert run_sampling_loop([{"type": "tool_call", "name": "delete", "args": {}}], {})["status"] == "invalid_step"


## 6. Form Elicitation：只允许受限平面 schema，禁止索取秘密

form 模式把结构化数据返回 client/server 链路，只适合非敏感信息。host 应验证 schema 是平面 primitive、字段数量/长度和请求频率，并明确显示哪个 server 在请求、用途以及 accept/decline/cancel。


In [ ]:
SENSITIVE_NAMES = {"password", "secret", "token", "api_key", "credit_card"}

def validate_form_schema(schema):
    if schema.get("type") != "object" or not isinstance(schema.get("properties"), dict):
        return False, "object_required"
    for name, definition in schema["properties"].items():
        if name.casefold() in SENSITIVE_NAMES:
            return False, "sensitive_field"
        if definition.get("type") not in {"string", "number", "integer", "boolean"}:
            return False, "unsupported_type"
    return True, "ok"

# 普通筛选字段通过；秘密字段与嵌套对象被拒绝。
form = {"type": "object", "properties": {"city": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["city"]}
assert validate_form_schema(form)[0]
assert validate_form_schema({"type": "object", "properties": {"api_key": {"type": "string"}}})[1] == "sensitive_field"
assert not validate_form_schema({"type": "object", "properties": {"profile": {"type": "object"}}})[0]


## 7. URL Elicitation：敏感流程走带上下文的站外交互

2025-11-25 引入 URL mode，适合第三方授权/支付等不应经过 MCP client 的数据。client 只展示 URL 与请求上下文；仍要限制 HTTPS、可信 host、唯一 elicitationId、防重放，并强调它不是 client 对 MCP server 的授权流程。


In [ ]:
def validate_url_elicitation(params, allowed_hosts, seen_ids):
    if params.get("mode") != "url" or not params.get("elicitationId"):
        return False, "missing_fields"
    parsed = urlsplit(params.get("url", ""))
    if parsed.scheme != "https" or parsed.hostname not in allowed_hosts:
        return False, "untrusted_url"
    if params["elicitationId"] in seen_ids:
        return False, "replay"
    return True, "show_to_user"

# HTTPS allowlist 首次请求通过；HTTP 与相同 id 重放失败。
url_request = {"mode": "url", "message": "连接代码仓库", "url": "https://auth.example/connect?state=opaque", "elicitationId": "el-7"}
assert validate_url_elicitation(url_request, {"auth.example"}, set())[0]
assert validate_url_elicitation({**url_request, "url": "http://auth.example/connect"}, {"auth.example"}, set())[1] == "untrusted_url"
assert validate_url_elicitation(url_request, {"auth.example"}, {"el-7"})[1] == "replay"


## 8. 审计与版本演进：按修订绑定 schema，并为弃用准备降级路径

规范在演进：2025-11-25 增加 sampling tools、URL elicitation 和实验 tasks，后续 draft/SEP 还可能软弃用或替换能力。实现不能只检查方法名；日志应记录协议修订、server identity、用户决定、预算、请求摘要和结果去向。


In [ ]:
@dataclass(frozen=True)
class AuditEvent:
    protocol_version: str
    server_id: str
    method: str
    request_id: int
    decision: str
    payload_hash: str

def make_audit(server_id, request):
    payload_hash = hashlib.sha256(json.dumps(request, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
    return AuditEvent(PROTOCOL_VERSION, server_id, request["method"], request["id"], "approved", payload_hash)

# 审计绑定修订和请求；payload 变化会生成不同摘要；敏感原文不直接进入事件字段。
event = make_audit("server.example/v3", request)
changed_event = make_audit("server.example/v3", {**request, "params": {"cursor": "next"}})
assert event.protocol_version == PROTOCOL_VERSION
assert event.payload_hash != changed_event.payload_hash
assert len(event.payload_hash) == 64


## 面试收束与生产替换点

完整回答不要停在算法名：先说清业务目标、输入输出与信任边界，再给核心数据结构/公式和可执行 oracle，最后落到离线切片、线上 SLO、成本、安全、版本、灰度与回滚。这里的受控实现用于解释机制和发现反例；真实模型编码器、分布式索引、协议 SDK、安全沙箱、监控与持久层应作为可替换组件，并用同一合同验收。

典型追问包括：数据规模扩大后瓶颈在哪？近似步骤损失了什么？哪个状态必须持久化？超时或部分失败怎样降级？版本错配为何不能静默兼容？离线指标上升是否来自污染、权限泄漏、评测器偏差或重复样本？
